# 🎧 ทดสอบ TTS ภาษาไทย — ฟังเทียบทีละตัว

**วิธีใช้ (ครั้งแรก):**
1. เมนู `Runtime` → `Change runtime type` → เลือก **T4 GPU** → Save
2. เมนู `Runtime` → `Run all` แล้วรอ ~5-10 นาที (โหลดโมเดลครั้งแรก)
3. เลื่อนลงล่างสุด → มีปุ่มกดฟังเสียงทุกตัวเทียบกัน

เอนจินที่ทดสอบ: **edge-tts** (3 เสียง Microsoft) / **MMS** (Meta + รุ่นเสียงหญิงปรับปรุง) / **F5-TTS-THAI** (โคลนเสียง)

ประโยคทดสอบ 6 แบบ: ทักทาย / เล่าเรื่อง / ตัวเลขดิบ (stress test) / อังกฤษปน (stress test) / ทับศัพท์ / อารมณ์+ไม้ยมก

In [ ]:
# 1) ติดตั้ง (~2-3 นาที)
!pip install -q edge-tts f5-tts-th soundfile
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ไม่มี — F5 จะช้ามาก แนะนำเปิด T4')

In [ ]:
# 2) ประโยคทดสอบมาตรฐาน (ชุดเดียวกับ tts-ai/tests/sentences_th.json ใน repo)
import os, time
os.makedirs('samples', exist_ok=True)

SENTENCES = {
  '01_basic': 'สวัสดีครับ ยินดีต้อนรับเข้าสู่ช่องของเรา วันนี้มีเรื่องราวน่าสนใจมาเล่าให้ฟังกันครับ',
  '02_story': 'ลองจินตนาการดูนะครับ ชายคนหนึ่งตัดสินใจเดินออกจากช่องยูทูบที่มีผู้ติดตามยี่สิบล้านคน โดยไม่บอกใครแม้แต่คำเดียว',
  '03_numbers_raw': 'เมื่อเวลา 18:30 น. ของวันที่ 2 สิงหาคม 2569 ยอดวิวพุ่งไปถึง 2,500,000 วิว สร้างรายได้กว่า 64,000 บาท',
  '04_english_mix': 'ช่อง YouTube ของเราใช้ AI ทำคลิป Shorts ลง TikTok และ Facebook ทุกวัน',
  '05_english_thai_spelled': 'ช่องยูทูบของเราใช้เอไอทำคลิปช็อตส์ ลงติ๊กต่อกและเฟซบุ๊กทุกวัน',
  '06_emotion': 'อะไรนะครับ! เขาทิ้งรายได้เดือนละหลายล้านบาทไปเฉย ๆ อย่างนั้นเหรอ เรื่องนี้มันบ้ามาก ๆ ไม่มีใครเชื่อเลยจริง ๆ',
}
timing = {}
print('พร้อม:', len(SENTENCES), 'ประโยค')

In [ ]:
# 3) edge-tts — เสียง Microsoft 3 เสียง (ฟรี เร็ว ไม่ใช้ GPU)
import edge_tts, asyncio

async def gen_edge():
    for voice in ['th-TH-PremwadeeNeural', 'th-TH-NiwatNeural', 'th-TH-AcharaNeural']:
        short = voice.replace('th-TH-', '').replace('Neural', '').lower()
        for sid, text in SENTENCES.items():
            t0 = time.time()
            await edge_tts.Communicate(text, voice).save(f'samples/edge_{short}_{sid}.mp3')
            timing[f'edge_{short}_{sid}'] = time.time() - t0
        print('เสร็จ:', voice)

await gen_edge()

In [ ]:
# 4) MMS — โมเดลเบาของ Meta + รุ่นเสียงหญิงปรับปรุง (รัน CPU ก็ได้)
import scipy.io.wavfile
from transformers import VitsModel, AutoTokenizer

for short, mid in {'meta-tha': 'facebook/mms-tts-tha', 'femalev2': 'VIZINTZOR/MMS-TTS-THAI-FEMALEV2'}.items():
    try:
        model = VitsModel.from_pretrained(mid)
        tok = AutoTokenizer.from_pretrained(mid)
        for sid, text in SENTENCES.items():
            t0 = time.time()
            inputs = tok(text, return_tensors='pt')
            with torch.no_grad():
                wav = model(**inputs).waveform
            scipy.io.wavfile.write(f'samples/mms_{short}_{sid}.wav', rate=model.config.sampling_rate, data=wav.squeeze().numpy())
            timing[f'mms_{short}_{sid}'] = time.time() - t0
        print('เสร็จ:', mid)
    except Exception as e:
        print('ข้าม', mid, '->', e)

In [ ]:
# 5) F5-TTS-THAI — ตัวหลัก (โคลนเสียง) ใช้เสียง Premwadee ประโยคแรกเป็นเสียงอ้างอิง
#    ถ้าอยากโคลนเสียงตัวเอง: อัปโหลดไฟล์ .wav ~10 วิ แล้วแก้ REF_AUDIO / REF_TEXT
import soundfile as sf
from f5_tts_th.tts import TTS

REF_AUDIO = 'samples/edge_premwadee_01_basic.mp3'
REF_TEXT = SENTENCES['01_basic']

tts = TTS(model='v1')  # ลอง 'v2' (IPA) ได้ — อ่านคำยากแม่นขึ้น
for sid, text in SENTENCES.items():
    t0 = time.time()
    wav = tts.infer(ref_audio=REF_AUDIO, ref_text=REF_TEXT, gen_text=text, step=32, cfg=2.0, speed=1.0)
    sf.write(f'samples/f5thai_v1_{sid}.wav', wav, 24000)
    timing[f'f5thai_v1_{sid}'] = time.time() - t0
    print(f'เสร็จ: {sid} ({timing[f"f5thai_v1_{sid}"]:.1f} วิ)')

In [ ]:
# 6) 🎧 ฟังเทียบทุกตัว — เรียงตามประโยค แต่ละประโยคมีทุกเอนจินให้กดฟังต่อกัน
import glob
from IPython.display import Audio, display, Markdown

for sid in SENTENCES:
    display(Markdown(f'---\n### {sid}\n> {SENTENCES[sid]}'))
    for f in sorted(glob.glob(f'samples/*_{sid}.*')):
        name = f.split('/')[-1]
        secs = timing.get(name.rsplit('.', 1)[0])
        display(Markdown(f'**{name}**' + (f' — {secs:.1f} วิ' if secs else '')))
        display(Audio(f))

In [ ]:
# 7) (ทางเลือก) รวมไฟล์ทั้งหมดเป็น zip ไว้โหลดเก็บ
!zip -q -r thai_tts_samples.zip samples
from google.colab import files
files.download('thai_tts_samples.zip')